### All Imports

In [ ]:
import pandas as pd
import numpy as np
import polars as pl
import json
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
lf = pl.scan_ndjson(
    "../data/raw/meta_Electronics.jsonl.gz",
    ignore_errors=True,
)

In [ ]:
schema = lf.collect_schema()
for name, dtype in schema.items():
    print(f"{name}: {dtype}")

In [ ]:
# Print the first raw
first_row = lf.head(1).collect().row(0, named=True)
print(json.dumps(first_row, indent=2, default=str))

In [ ]:
# Filter products first available in 2022 or 2023
year = (
    pl.col("details")
    .struct.field("Date First Available")
    .str.extract(r"(\d{4})$")
    .cast(pl.Int32)
)

meta_electronics_2022_2023 = lf.filter(year.is_in([2022, 2023])).collect()
print(f"Count: {meta_electronics_2022_2023.height}")

In [ ]:
# Drop rows where main_category is null
meta_electronics_2022_2023 = meta_electronics_2022_2023.filter(
    pl.col("main_category").is_not_null()
)
print(f"Count after dropping null main_category: {meta_electronics_2022_2023.height}")

In [ ]:
# Explore distribution of main_category using pandas
category_counts = (
    meta_electronics_2022_2023["main_category"]
    .to_pandas()
    .value_counts()
)

category_counts.plot.bar(
    figsize=(10, 6),
    title="Distribution of main_category (2022–2023)",
    xlabel="Main Category",
    ylabel="Count",
)
plt.tight_layout()
plt.show()

In [ ]:
# Keep products with at least 100 ratings
meta_electronics_2022_2023_ratting_100 = meta_electronics_2022_2023.filter(
    pl.col("rating_number") >= 100
)
print(f"Count after rating_number >= 100: {meta_electronics_2022_2023_ratting_100.height}")

In [ ]:
# Explore distribution of main_category using pandas
category_counts = (
    meta_electronics_2022_2023_ratting_100["main_category"]
    .to_pandas()
    .value_counts()
)

category_counts.plot.bar(
    figsize=(10, 6),
    title="Distribution of main_category (2022–2023)",
    xlabel="Main Category",
    ylabel="Count",
)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of average_rating
meta_electronics_2022_2023_ratting_100["average_rating"].to_pandas().plot.hist(
    bins=100,
    range=(0, 5),
    figsize=(10, 6),
    title="Distribution of average_rating (2022–2023, >=100 ratings)",
    xlabel="Average Rating",
    ylabel="Count",
)
plt.tight_layout()
plt.show()

In [ ]:
# Random sample of 1000 products
meta_electronics_2022_2023_ratting_100_sample_1000 = (
    meta_electronics_2022_2023_ratting_100.sample(n=1000, seed=42)
)
print(f"Sample size: {meta_electronics_2022_2023_ratting_100_sample_1000.height}")

In [ ]:
# Distribution of average_rating
meta_electronics_2022_2023_ratting_100_sample_1000["average_rating"].to_pandas().plot.hist(
    bins=100,
    range=(0, 5),
    figsize=(10, 6),
    title="Distribution of average_rating (2022–2023, >=100 ratings)",
    xlabel="Average Rating",
    ylabel="Count",
)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of price
meta_electronics_2022_2023_ratting_100_sample_1000["price"].to_pandas().plot.hist(
    bins=100,
    range=(0, 500),
    figsize=(10, 6),
    title="Distribution of price (2022–2023, >=100 ratings)",
    xlabel="Price",
    ylabel="Count",
)
plt.tight_layout()
plt.show()

In [ ]:
# Save the sample to newline-delimited JSON (record-oriented)

output_path = Path("../data/processed/meta_electronics_2022_2023_ratting_100_sample_1000.jsonl")
output_path.parent.mkdir(parents=True, exist_ok=True)

meta_electronics_2022_2023_ratting_100_sample_1000.write_ndjson(output_path)
print(f"Saved {meta_electronics_2022_2023_ratting_100_sample_1000.height} records to {output_path}")

In [ ]:
# Find matching reviews for the sampled products (join on parent_asin)
sample_asins = meta_electronics_2022_2023_ratting_100_sample_1000["parent_asin"].unique()

reviews_lf = pl.scan_ndjson(
    "../data/raw/Electronics.jsonl.gz",
    ignore_errors=True,
)

review_electronics_2022_2023_ratting_100_sample_1000 = reviews_lf.filter(
    pl.col("parent_asin").is_in(sample_asins.to_list())
).collect()

print(f"Matching reviews: {review_electronics_2022_2023_ratting_100_sample_1000.height}")

In [ ]:
# Save the matching reviews to newline-delimited JSON (record-oriented)
review_output_path = Path("../data/processed/review_electronics_2022_2023_ratting_100_sample_1000.jsonl")
review_output_path.parent.mkdir(parents=True, exist_ok=True)

review_electronics_2022_2023_ratting_100_sample_1000.write_ndjson(review_output_path)
print(f"Saved {review_electronics_2022_2023_ratting_100_sample_1000.height} records to {review_output_path}")